#Punto 2. Analisis de Calidad

Analisis de calidad de los datos desde varias perspectivas. Revision de datos default o sentinel. 

No se realiza limpieza o modificaciones de los datos


# 0 · Lectura (mismo esquema que la ingesta)

`spark.read` es perezoso: el esquema se declara, no se infiere.

In [0]:
# ──── 0 · librerías, rutas, esquema ────
from pyspark.sql import functions as f

CSV_DIR = "/Volumes/workspace/default/oceanwatch/raw/csv"

AIS_SCHEMA = """
    MMSI long,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG double,
    COG double,
    Heading double,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType int,
    Status int,
    Length int,
    Width int,
    Draft double,
    Cargo int,
    TransceiverClass string
"""

ais = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(AIS_SCHEMA)
    .csv(CSV_DIR)
    .withColumn("archivo", f.element_at(f.split(f.col("_metadata.file_path"), "/"), -1))
    .withColumn("fecha", f.to_date("BaseDateTime"))
)

ais.printSchema()

root
 |-- MMSI: long (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: double (nullable = true)
 |-- COG: double (nullable = true)
 |-- Heading: double (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- Length: integer (nullable = true)
 |-- Width: integer (nullable = true)
 |-- Draft: double (nullable = true)
 |-- Cargo: integer (nullable = true)
 |-- TransceiverClass: string (nullable = true)
 |-- archivo: string (nullable = true)
 |-- fecha: date (nullable = true)



# 1 · Perfil

El flujo de la semana 5: cuánto hay, mirar filas, agrupar.

## 1.1 Primer contacto
`display` trae solo lo justo para inspeccionar.

In [0]:
# ──── 1.1 · muestra ────
display(ais)

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,archivo,fecha
367138690,2023-06-01T00:00:01.000Z,30.28335,-91.22283,0.0,273.0,511.0,ST FRANCISVILLE,null,WDD3968,60,0,43,18,null,60,A,AIS_2023_06_01.csv,2023-06-01
368036960,2023-06-01T00:00:01.000Z,40.49818,-80.07596,3.3,118.6,131.0,GALE R RHODES,IMO8199788,WDK2590,0,12,20,7,2.6,0,A,AIS_2023_06_01.csv,2023-06-01
368095340,2023-06-01T00:00:01.000Z,29.99678,-93.94959,0.0,309.8,511.0,GAMBLER,null,WDK8562,52,15,19,8,0.0,52,A,AIS_2023_06_01.csv,2023-06-01
248301000,2023-06-01T00:00:03.000Z,26.13342,-80.11909,0.0,260.5,86.0,ATLAS,null,9HB5599,37,5,31,8,3.0,37,A,AIS_2023_06_01.csv,2023-06-01
367186410,2023-06-01T00:00:03.000Z,30.00762,-93.95571,5.7,164.0,164.0,BRIAN O DANIELS,null,WDD7438,31,12,26,9,null,31,A,AIS_2023_06_01.csv,2023-06-01
367108820,2023-06-01T00:00:05.000Z,37.82833,-76.27936,0.1,235.2,511.0,TIDELANDS,IMO7309443,WX5026,30,0,61,9,null,30,A,AIS_2023_06_01.csv,2023-06-01
367320740,2023-06-01T00:00:05.000Z,38.73458,-82.87265,4.7,350.3,349.0,TENNESSEE,IMO8836584,WDE2124,31,0,42,12,null,52,A,AIS_2023_06_01.csv,2023-06-01
368101220,2023-06-01T00:00:03.000Z,32.6922,-117.14808,0.0,360.0,511.0,WILLIAM F,null,WDK9183,33,15,40,18,0.0,33,A,AIS_2023_06_01.csv,2023-06-01
367529860,2023-06-01T00:00:08.000Z,29.61437,-95.00393,0.2,197.6,285.0,C R MCCASKILL,null,WDG3808,90,3,67,null,null,33,A,AIS_2023_06_01.csv,2023-06-01
367558580,2023-06-01T00:00:09.000Z,27.3326,-82.54605,0.0,233.2,221.0,DAYS LIKE THIS,null,WDG6586,37,0,40,7,null,37,A,AIS_2023_06_01.csv,2023-06-01


## 1.2 Posiciones por día
Una fila AIS = una posición. El agrupado por `fecha` (del `BaseDateTime`)
responde el primer punto de la rúbrica.

In [0]:
# ──── 1.2 · posiciones por día ────
posiciones_dia = (
    ais
    .groupBy("fecha")
    .agg(f.count("*").alias("posiciones"))
    .orderBy("fecha")
)

display(posiciones_dia)

fecha,posiciones
2023-06-01,8808904
2023-06-02,9052241
2023-06-03,8036348
2023-06-04,8522645
2023-06-05,8613757
2023-06-06,8587938
2023-06-07,8911726


## 1.3 Buques únicos
`MMSI` es el identificador del transmisor. `countDistinct` es el conteo exacto

In [0]:
# ──── 1.3 · buques distintos (exacto) ────
unicos = ais.agg(
    f.count("*").alias("posiciones"),
    f.countDistinct("MMSI").alias("buques_distintos"),
    f.countDistinct("fecha").alias("dias"),
)

display(unicos)

posiciones,buques_distintos,dias
60533559,31871,7


## 1.4 Tráfico por tipo de buque
`VesselType` es el código AIS (el catálogo con nombres se cruza más adelante).
`0` suele ser “no disponible”.

In [0]:
# ──── 1.4 · distribución VesselType ────
por_tipo = (
    ais
    .groupBy("VesselType")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques"),
        f.round(f.avg("SOG"), 2).alias("sog_media"),
    )
    .orderBy(f.desc("posiciones"))
)

display(por_tipo)

VesselType,posiciones,buques,sog_media
31,16558120,3451,1.9
37,14476696,12934,1.64
60,4795209,1708,4.43
30,4010188,2192,2.47
36,3876264,4243,1.98
90,3848162,1532,2.03
70,3837926,1563,6.37
52,1921565,482,2.09
80,1789168,818,5.87
57,1323765,242,1.91


## 1.5 Tamaño de los buques
`Length`, `Width`, `Draft` vienen en el mensaje estático. Ceros y nulos son
frecuentes, demostrando una oportunidad de limpieza

In [0]:
# ──── 1.5 · Length / Width / Draft ────
display(ais.select("Length", "Width", "Draft").summary())

nulos_tamano = ais.agg(
    f.sum(f.col("Length").isNull().cast("int")).alias("length_nulo"),
    f.sum((f.col("Length") == 0).cast("int")).alias("length_cero"),
    f.sum(f.col("Width").isNull().cast("int")).alias("width_nulo"),
    f.sum((f.col("Width") == 0).cast("int")).alias("width_cero"),
    f.sum(f.col("Draft").isNull().cast("int")).alias("draft_nulo"),
    f.sum((f.col("Draft") == 0).cast("int")).alias("draft_cero"),
)

display(nulos_tamano)

summary,Length,Width,Draft
count,56868834,51313992,21532694
mean,48.62184200928051,10.767158555896412,5.013947980692658
stddev,67.96676908682359,9.824210086365955,4.0979379819787685
min,0,0,0.0
25%,15,5,2.4
50%,22,8,3.7
75%,38,11,7.3
max,638,126,25.5


length_nulo,length_cero,width_nulo,width_cero,draft_nulo,draft_cero
3664725,2079143,9219567,2227285,39000865,2719462


# 2 · Problemas de calidad

Datos de sensores: sentinels del estándar AIS, rangos imposibles, identificadores
raros, duplicados. Cada flag es una columna derivada (`when`/`otherwise`);
después se agregan conteos. **No se filtra el dataset.**

Referencia de sentinels AIS: Los valores sentinels son una forma de decir “not available” del estándar de radio
(ITU-R M.1371), no del diccionario NOAA. Tabla pública (USCG):
[Class A AIS Position Report (Messages 1, 2, and 3)](https://navcen.uscg.gov/ais-class-a-reports)

| campo | valor típico de “no disponible” |
|---|---|
| LAT | 91 |
| LON | 181 |
| SOG | 102.3 |
| COG | 360 |
| Heading | 511 |

Un buque mercante por encima de 50 nudos es físicamente dudoso (no es el
sentinel, pero sí una anomalía a revisar en Entrega 2).

In [0]:
# ──── 2 · flags de calidad (transformaciones, aún no se ejecutan) ────
calidad = (
    ais
    .withColumn(
        "mmsi_anomalo", # Id del transmisor no valido.
        (f.col("MMSI").isNull())
        | (f.col("MMSI") < 100000000)
        | (f.col("MMSI") > 999999999),
    )
    .withColumn(
        "lat_fuera_rango", # Latitud fuera de rango.
        f.col("LAT").isNull()
        | (f.col("LAT") < -90)
        | (f.col("LAT") > 90)
        | (f.abs(f.col("LAT")) >= 91),
    )
    .withColumn(
        "lon_fuera_rango", # Longitud fuera de rango.
        f.col("LON").isNull()
        | (f.col("LON") < -180)
        | (f.col("LON") > 180)
        | (f.abs(f.col("LON")) >= 181),
    )
    .withColumn(
        "sog_sentinel", # Velocidad sobre el guarismo no valido.
        f.col("SOG").isNull() | (f.col("SOG") < 0) | (f.col("SOG") >= 102.2),
    )
    .withColumn(
        "sog_sospechosa", # Velocidad sospechosa.
        (f.col("SOG") > 50) & (f.col("SOG") < 102.2),
    )
    .withColumn(
        "cog_sentinel", # Rumbo no valido,  medido en grados.
        f.col("COG").isNull() | (f.col("COG") < 0) | (f.col("COG") >= 360),
    )
    .withColumn(
        "heading_sentinel", # Direccion no valida, medida en grados
        f.col("Heading").isNull()
        | (f.col("Heading") < 0)
        | (f.col("Heading") > 359),
    )
    .withColumn(
        "nombre_vacio", # Nombre del buque no valido.
        f.col("VesselName").isNull() | (f.trim(f.col("VesselName")) == ""),
    )
    .withColumn(
        "imo_vacio", # Id del buque no valido.
        f.col("IMO").isNull() | (f.trim(f.col("IMO")) == ""),
    )
    .withColumn(
        "tipo_nulo_o_cero", # Tipo de buque no valido.
        f.col("VesselType").isNull() | (f.col("VesselType") == 0),
    )
    .withColumn(
        "fecha_fuera_semana", # Fecha fuera de la semana de la base de datos.
        f.col("fecha").isNull()
        | (f.col("fecha") < f.lit("2023-06-01").cast("date"))
        | (f.col("fecha") > f.lit("2023-06-07").cast("date")),
    )
)

## 2.1 Conteos y porcentajes
Una sola agregación recorre el corpus y deja el tablero de calidad.

In [0]:
# ──── 2.1 · tablero ────
n = f.count("*")

tablero = calidad.agg( #Usamos agregacion para sumar, no hacemos filtrado en el dataset.
    n.alias("posiciones"),
    f.sum(f.col("mmsi_anomalo").cast("int")).alias("mmsi_anomalo"),
    f.sum(f.col("lat_fuera_rango").cast("int")).alias("lat_fuera_rango"),
    f.sum(f.col("lon_fuera_rango").cast("int")).alias("lon_fuera_rango"),
    f.sum(f.col("sog_sentinel").cast("int")).alias("sog_sentinel"),
    f.sum(f.col("sog_sospechosa").cast("int")).alias("sog_sospechosa"),
    f.sum(f.col("cog_sentinel").cast("int")).alias("cog_sentinel"),
    f.sum(f.col("heading_sentinel").cast("int")).alias("heading_sentinel"),
    f.sum(f.col("nombre_vacio").cast("int")).alias("nombre_vacio"),
    f.sum(f.col("imo_vacio").cast("int")).alias("imo_vacio"),
    f.sum(f.col("tipo_nulo_o_cero").cast("int")).alias("tipo_nulo_o_cero"),
    f.sum(f.col("fecha_fuera_semana").cast("int")).alias("fecha_fuera_semana"),
)

# una fila ancha → una fila por problema (más fácil de leer y de citar)
problemas = (
    tablero
    .select(
        f.explode(
            f.array(
                f.struct(f.lit("mmsi_anomalo").alias("problema"), f.col("mmsi_anomalo").alias("filas")),
                f.struct(f.lit("lat_fuera_rango").alias("problema"), f.col("lat_fuera_rango").alias("filas")),
                f.struct(f.lit("lon_fuera_rango").alias("problema"), f.col("lon_fuera_rango").alias("filas")),
                f.struct(f.lit("sog_sentinel").alias("problema"), f.col("sog_sentinel").alias("filas")),
                f.struct(f.lit("sog_sospechosa").alias("problema"), f.col("sog_sospechosa").alias("filas")),
                f.struct(f.lit("cog_sentinel").alias("problema"), f.col("cog_sentinel").alias("filas")),
                f.struct(f.lit("heading_sentinel").alias("problema"), f.col("heading_sentinel").alias("filas")),
                f.struct(f.lit("nombre_vacio").alias("problema"), f.col("nombre_vacio").alias("filas")),
                f.struct(f.lit("imo_vacio").alias("problema"), f.col("imo_vacio").alias("filas")),
                f.struct(f.lit("tipo_nulo_o_cero").alias("problema"), f.col("tipo_nulo_o_cero").alias("filas")),
                f.struct(f.lit("fecha_fuera_semana").alias("problema"), f.col("fecha_fuera_semana").alias("filas")),
            )
        ).alias("p"),
        f.col("posiciones"),
    )
    .select(
        "p.problema",
        "p.filas",
        "posiciones",
        f.round(f.col("p.filas") / f.col("posiciones") * 100, 4).alias("pct"),
    )
    .orderBy(f.desc("filas"))
)

display(problemas)

problema,filas,posiciones,pct
heading_sentinel,33536689,60533559,55.4018
imo_vacio,25891448,60533559,42.7721
cog_sentinel,10291143,60533559,17.0007
tipo_nulo_o_cero,925197,60533559,1.5284
sog_sentinel,160274,60533559,0.2648
nombre_vacio,150685,60533559,0.2489
mmsi_anomalo,49897,60533559,0.0824
sog_sospechosa,1144,60533559,0.0019
lat_fuera_rango,0,60533559,0.0
lon_fuera_rango,0,60533559,0.0


## 2.2 Duplicados
Misma llave de mensaje que indica: `MMSI` + `BaseDateTime`. Un `groupBy` que se queda
solo con las llaves que aparecen más de una vez.

In [0]:
# ──── 2.2 · duplicados MMSI + instante ────
dup_llaves = (
    ais
    .groupBy("MMSI", "BaseDateTime")
    .agg(f.count("*").alias("veces"))
    .filter(f.col("veces") > 1)
)

n_llaves_dup = dup_llaves.count()
n_filas_dup = dup_llaves.agg(f.coalesce(f.sum("veces"), f.lit(0)).alias("s")).first()["s"]

print(f"llaves (MMSI, BaseDateTime) repetidas: {n_llaves_dup:,}")
print(f"filas involucradas:                    {n_filas_dup:,}")

display(dup_llaves.orderBy(f.desc("veces")).limit(20))

llaves (MMSI, BaseDateTime) repetidas: 1,672
filas involucradas:                    3,344


MMSI,BaseDateTime,veces
368006020,2023-06-02T07:00:00.000Z,2
319952000,2023-06-02T03:00:00.000Z,2
368089050,2023-06-02T18:59:59.000Z,2
316043514,2023-06-02T04:00:00.000Z,2
339928000,2023-06-02T21:59:59.000Z,2
311001077,2023-06-02T23:59:59.000Z,2
338352384,2023-06-02T00:00:00.000Z,2
367309480,2023-06-02T09:00:00.000Z,2
538010199,2023-06-02T00:00:00.000Z,2
338192665,2023-06-02T19:59:59.000Z,2


## 2.3 Muestra de cada anomalía
No para “arreglar”, sino para ver cómo se ve la data de entrada, sin haber hecho ninguna limpieza sobre los datos.

Es interesante ver que no aparecen escenarios donde la ubicacion geografica este fuera de rango.

In [0]:
# ──── 2.3 · ejemplos (tope 20) ────
display(calidad.filter("mmsi_anomalo").select("MMSI", "BaseDateTime", "LAT", "LON", "VesselName").limit(20))
display(calidad.filter("lat_fuera_rango OR lon_fuera_rango").select("MMSI", "BaseDateTime", "LAT", "LON").limit(20))
display(calidad.filter("sog_sentinel OR sog_sospechosa").select("MMSI", "BaseDateTime", "SOG", "VesselType").limit(20))
display(calidad.filter("heading_sentinel").select("MMSI", "Heading", "COG").limit(20))

MMSI,BaseDateTime,LAT,LON,VesselName
69918000,2023-06-02T00:00:01.000Z,30.39216,-81.41233,US GOV VESSEL 87
3669884,2023-06-02T00:00:00.000Z,41.25619,-72.00913,null
3669883,2023-06-02T00:00:08.000Z,41.34561,-72.09588,null
4566544,2023-06-02T00:00:03.000Z,36.18688,-75.19316,null
69918000,2023-06-02T00:01:11.000Z,30.39219,-81.41223,US GOV VESSEL 87
2981331,2023-06-02T00:00:10.000Z,29.35621,-89.30001,JOHN JAMES CHARLES
1,2023-06-02T00:01:06.000Z,49.1253,-123.08509,A
3669884,2023-06-02T00:01:10.000Z,41.25619,-72.00913,null
3669883,2023-06-02T00:01:19.000Z,41.34559,-72.09589,null
63280801,2023-06-02T00:01:38.000Z,33.75038,-118.29794,3280801 10%


MMSI,BaseDateTime,LAT,LON


MMSI,BaseDateTime,SOG,VesselType
367640910,2023-06-02T00:00:05.000Z,102.3,60
316005667,2023-06-02T00:00:05.000Z,102.3,30
368119660,2023-06-02T00:00:05.000Z,102.3,57
368228250,2023-06-02T00:00:04.000Z,102.3,90
367503760,2023-06-02T00:00:00.000Z,102.3,31
232024857,2023-06-02T00:00:06.000Z,102.3,99
368282170,2023-06-02T00:00:00.000Z,102.3,37
368300680,2023-06-02T00:00:07.000Z,102.3,36
368199910,2023-06-02T00:00:45.000Z,102.3,37
368251560,2023-06-02T00:00:06.000Z,102.3,31


MMSI,Heading,COG
367614020,511.0,243.4
367372810,511.0,359.4
367148660,511.0,297.1
367113120,511.0,298.1
367711450,511.0,360.0
367300910,511.0,120.1
367776380,511.0,218.5
366771550,511.0,141.8
368288170,511.0,5.4
367602520,511.0,22.1


## 2.4 Plan de la agregación de calidad

Evidencia de que es un barrido con proyección de flags, no un UDF fila a fila.
En serverless no hay caché: el `count` de duplicados es **otra** lectura.

In [0]:
# ──── 2.4 · plan ────
tablero.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (13)
+- == Initial Plan ==
   PhotonResultStage (12)
   +- PhotonColumnarToRow (11)
      +- PhotonAgg (10)
         +- PhotonShuffleExchangeSource (9)
            +- PhotonShuffleMapStage (8)
               +- PhotonShuffleExchangeSink (7)
                  +- PhotonAgg (6)
                     +- PhotonProject (5)
                        +- PhotonProject (4)
                           +- PhotonProject (3)
                              +- PhotonRowToColumnar (2)
                                 +- Scan csv  (1)


(1) Scan csv 
Output [16]: [MMSI#11255L, BaseDateTime#11256, LAT#11257, LON#11258, SOG#11259, COG#11260, Heading#11261, VesselName#11262, IMO#11263, VesselType#11265, file_path#12455, file_name#12456, file_size#12457L, file_block_start#12458L, file_block_length#12459L, file_modification_time#12460]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/oceanwatch/raw/csv]
ReadSchema: struct<MMSI:bigint,BaseDateTime:ti

# Cierre

Completado el requisito 2: posiciones por día, buques únicos, tipo y tamaño,
más un tablero de calidad con conteo y porcentaje.

**No se escribió una tabla “limpia”.** Las reglas (qué se descarta, qué se
imputa, qué se deja) van en la Entrega 2.
